In [1]:
print("My setup is working!")

My setup is working!


In [2]:
import pandas as pd
data = pd.read_csv('official_data_en.csv', sep=';')
data.head()

,"oblast,raion,hromada,level,started_at,finished_at,source"
0,"Vinnytska oblast,,,oblast,2022-03-15 16:10:34+..."
1,"Zhytomyrska oblast,,,oblast,2022-03-15 16:11:2..."
2,"Cherkaska oblast,Umanskyi raion,Umanska teryto..."
3,"Mykolaivska oblast,Pervomaiskyi raion,m. Pervo..."
4,"Kirovohradska oblast,,,oblast,2022-03-15 16:15..."


In [3]:
import pandas as pd
df = pd.read_csv('official_data_en.csv', sep=';')
df['start'] = pd.to_datetime(df['start'])
df['end'] = pd.to_datetime(df['end'])
df['duration_minutes'] = (df['end'] - df['start']).dt.total_seconds() / 60
df[['start', 'end', 'duration_minutes']].head()

KeyError: 'start'

In [4]:
import pandas as pd
df = pd.read_csv('official_data_en.csv', sep=',')

df['start'] = pd.to_datetime(df['start'])
df['end'] = pd.to_datetime(df['end'])


df['duration_minutes'] = (df['end'] - df['start']).dt.total_seconds() / 60


df[['start', 'end', 'duration_minutes']].head()

KeyError: 'start'

In [5]:
import pandas as pd

df = pd.read_csv('official_data_en.csv', sep=',')

print("--- EXACT COLUMN NAMES ---")
print(df.columns.tolist())
print("\n") 

print("--- DATA FORMATS ---")
print(df.dtypes)
print("\n")

print("--- DATA PREVIEW ---")
display(df.head(3))

--- EXACT COLUMN NAMES ---
['oblast', 'raion', 'hromada', 'level', 'started_at', 'finished_at', 'source']


--- DATA FORMATS ---
oblast         str
raion          str
hromada        str
level          str
started_at     str
finished_at    str
source         str
dtype: object


--- DATA PREVIEW ---


,oblast,raion,hromada,level,started_at,finished_at,source
0,Vinnytska oblast,NaN,NaN,oblast,2022-03-15 16:10:34+00:00,2022-03-15 16:50:07+00:00,official
1,Zhytomyrska oblast,NaN,NaN,oblast,2022-03-15 16:11:25+00:00,2022-03-15 16:54:23+00:00,official
2,Cherkaska oblast,Umanskyi raion,Umanska terytorialna hromada,hromada,2022-03-15 16:11:50+00:00,2022-03-15 16:54:47+00:00,official


In [6]:
import pandas as pd

df = pd.read_csv('official_data_en.csv', sep=',')

df['started_at'] = pd.to_datetime(df['started_at'])
df['finished_at'] = pd.to_datetime(df['finished_at'])

df['duration_minutes'] = (df['finished_at'] - df['started_at']).dt.total_seconds() / 60

display(df[['started_at', 'finished_at', 'duration_minutes']].head())

,started_at,finished_at,duration_minutes
0,2022-03-15 16:10:34+00:00,2022-03-15 16:50:07+00:00,39.550000
1,2022-03-15 16:11:25+00:00,2022-03-15 16:54:23+00:00,42.966667
2,2022-03-15 16:11:50+00:00,2022-03-15 16:54:47+00:00,42.950000
3,2022-03-15 16:14:46+00:00,2022-03-15 16:57:08+00:00,42.366667
4,2022-03-15 16:15:11+00:00,2022-03-15 16:54:52+00:00,39.683333


In [7]:
region_durations = df.groupby('oblast')['duration_minutes'].sum()

region_hours = region_durations / 60

region_ranking = region_hours.sort_values(ascending=False)

display(region_ranking.head(10))

oblast
Dnipropetrovska oblast    153466.170556
Kharkivska oblast         109223.067500
Donetska oblast            91503.528889
Zaporizka oblast           44589.892500
Sumska oblast              41468.895000
Chernihivska oblast        30648.635000
Poltavska oblast           19329.778333
Mykolaivska oblast         15742.642500
Cherkaska oblast           14863.874167
Kyivska oblast             14740.105278
Name: duration_minutes, dtype: float64

In [8]:
df['start_hour'] = df['started_at'].dt.hour

hourly_counts = df['start_hour'].value_counts().sort_index()

display(hourly_counts)

start_hour
0      9786
1      8397
2      6746
3      5622
4      7387
5      9013
6     12866
7     13776
8     12818
9     13186
10    13771
11    12911
12    13698
13    11651
14    12036
15    10991
16    11437
17    12864
18    14995
19    13571
20    11443
21    11651
22    11037
23    10241
Name: count, dtype: int64

In [9]:
duplicate_count = df.duplicated().sum()
print(f"Total exact duplicate rows: {duplicate_count}")

df = df.drop_duplicates()

print("\n--- HOW ALERTS ARE CATEGORIZED ---")
print(df['level'].value_counts())

Total exact duplicate rows: 113845

--- HOW ALERTS ARE CATEGORIZED ---
level
raion      73655
oblast     65128
hromada    19266
Name: count, dtype: int64


In [10]:
sample_alert = df[df['level'] == 'oblast'].iloc[0]
target_region = sample_alert['oblast']
target_time = sample_alert['started_at']

print(f"--- INVESTIGATING: {target_region} at exactly {target_time} ---")

simultaneous_alerts = df[(df['oblast'] == target_region) & (df['started_at'] == target_time)]

display(simultaneous_alerts[['oblast', 'raion', 'hromada', 'level']])

--- INVESTIGATING: Vinnytska oblast at exactly 2022-03-15 16:10:34+00:00 ---


,oblast,raion,hromada,level
0,Vinnytska oblast,NaN,NaN,oblast


In [11]:
region_name = 'Kyivska oblast'

major_alert = df[(df['oblast'] == region_name) & (df['level'] == 'oblast')].iloc[0]

start_time = major_alert['started_at']
end_time = major_alert['finished_at']

print(f"--- THE MAJOR ALERT TESTED ---")
print(f"Region: {region_name} | Level: OBLAST")
print(f"Started:  {start_time}")
print(f"Finished: {end_time}\n")

overlapping_alerts = df[
    (df['oblast'] == region_name) & 
    (df['level'] != 'oblast') & 
    (df['started_at'] >= start_time) & 
    (df['started_at'] <= end_time)
]

print(f"--- SMALLER ALERTS DURING THIS EXACT TIME ---")
print(f"Found {len(overlapping_alerts)} overlapping alerts triggered a few seconds/minutes later.")

if len(overlapping_alerts) > 0:
    display(overlapping_alerts[['oblast', 'raion', 'hromada', 'level', 'started_at', 'finished_at']].head())

--- THE MAJOR ALERT TESTED ---
Region: Kyivska oblast | Level: OBLAST
Started:  2022-03-15 18:06:20+00:00
Finished: 2022-03-15 19:12:29+00:00

--- SMALLER ALERTS DURING THIS EXACT TIME ---
Found 0 overlapping alerts triggered a few seconds/minutes later.


In [12]:
import pandas as pd

df_raw = pd.read_csv('official_data_en.csv', sep=',')

duplicates = df_raw[df_raw.duplicated(keep=False)]

sorted_duplicates = duplicates.sort_values(by=['oblast', 'started_at'])

print(f"Total rows involved in duplication: {len(sorted_duplicates)}")
print("\n--- LOOK CLOSELY AT THESE PAIRS ---")
display(sorted_duplicates.head(6))

Total rows involved in duplication: 227690

--- LOOK CLOSELY AT THESE PAIRS ---


,oblast,raion,hromada,level,started_at,finished_at,source
2,Cherkaska oblast,Umanskyi raion,Umanska terytorialna hromada,hromada,2022-03-15 16:11:50+00:00,2022-03-15 16:54:47+00:00,official
113913,Cherkaska oblast,Umanskyi raion,Umanska terytorialna hromada,hromada,2022-03-15 16:11:50+00:00,2022-03-15 16:54:47+00:00,official
24,Cherkaska oblast,Umanskyi raion,Umanska terytorialna hromada,hromada,2022-03-15 18:04:27+00:00,2022-03-15 18:35:27+00:00,official
113935,Cherkaska oblast,Umanskyi raion,Umanska terytorialna hromada,hromada,2022-03-15 18:04:27+00:00,2022-03-15 18:35:27+00:00,official
25,Cherkaska oblast,Umanskyi raion,Zhashkivska terytorialna hromada,hromada,2022-03-15 18:04:36+00:00,2022-03-15 18:35:23+00:00,official
113936,Cherkaska oblast,Umanskyi raion,Zhashkivska terytorialna hromada,hromada,2022-03-15 18:04:36+00:00,2022-03-15 18:35:23+00:00,official


In [13]:
import pandas as pd

# 1. Load the raw data
df = pd.read_csv('official_data_en.csv', sep=',')

# 2. Permanently drop the exact duplicates
df = df.drop_duplicates()

# 3. Convert the text columns into live digital clocks
df['started_at'] = pd.to_datetime(df['started_at'])
df['finished_at'] = pd.to_datetime(df['finished_at'])

# 4. Calculate the exact duration in minutes
df['duration_minutes'] = (df['finished_at'] - df['started_at']).dt.total_seconds() / 60

# 5. Extract the Hour and the Year-Month for our future charts
df['start_hour'] = df['started_at'].dt.hour
df['year_month'] = df['started_at'].dt.to_period('M').astype(str)

# 6. Verify the clean state
print(f"Data cleaning successful! Total unique rows ready for analysis: {len(df)}")

Data cleaning successful! Total unique rows ready for analysis: 158049


/var/folders/6p/4z3wt6c97pn8q43tk9gnzqy40000gn/T/ipykernel_58989/2984369319.py:18: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['year_month'] = df['started_at'].dt.to_period('M').astype(str)


In [14]:
# Show the first 5 rows of our clean dataset with all columns
display(df.head())

,oblast,raion,hromada,level,started_at,finished_at,source,duration_minutes,start_hour,year_month
0,Vinnytska oblast,NaN,NaN,oblast,2022-03-15 16:10:34+00:00,2022-03-15 16:50:07+00:00,official,39.550000,16,2022-03
1,Zhytomyrska oblast,NaN,NaN,oblast,2022-03-15 16:11:25+00:00,2022-03-15 16:54:23+00:00,official,42.966667,16,2022-03
2,Cherkaska oblast,Umanskyi raion,Umanska terytorialna hromada,hromada,2022-03-15 16:11:50+00:00,2022-03-15 16:54:47+00:00,official,42.950000,16,2022-03
3,Mykolaivska oblast,Pervomaiskyi raion,m. Pervomaisk ta Pervomaiska terytorialna hromada,hromada,2022-03-15 16:14:46+00:00,2022-03-15 16:57:08+00:00,official,42.366667,16,2022-03
4,Kirovohradska oblast,NaN,NaN,oblast,2022-03-15 16:15:11+00:00,2022-03-15 16:54:52+00:00,official,39.683333,16,2022-03


In [15]:
# Export our clean, finalized dataframe to a new physical file
df.to_csv('clean_air_raid_data.csv', index=False)

print("Clean data safely exported to your folder!")

Clean data safely exported to your folder!
